In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")

matplotlib.rcParams['font.family'] = 'serif'
matplotlib.rcParams['font.serif'] = ['Times New Roman']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ===============================
# Paths
# ===============================
data_path = r"E:\关中干旱\index\final_indices_for_clustering.csv"
out_png   = r"E:\关中干旱\index\radar_density_v2.png"
out_pdf   = r"E:\关中干旱\index\radar_density_v2.pdf"

# ===============================
# Labels & Config
# ===============================
labels   = ["EAT", "SCSH", "WPSH", "AO-S", "AO-W"]
csv_cols = ["东亚槽", "南海副高面积", "西太平洋副高北界", "北极涛动", "冬季北极涛动"]

YEAR_COL    = "Year"
SPI3_COL    = "spring_SPI3"
CLUSTER_COL = "SOM_Cluster"

cluster_colors = {
    0: "#FE2400",
    1: "#64D000",
    2: "#833198"
}
cluster_roman = {
    0: "I",
    1: "II",
    2: "III"
}

V_MIN, V_MAX = -3.0, 3.0

# ===============================
# Style Constants
# ===============================
EVENT_LW = 2.6
EVENT_ALPHA = 0.88

DENSITY_BINS = 40
ALPHA_BASE   = 0.10

# 背景色：略加深
BAND_BLUE  = "#C7E3FF"   # -1 到 0
BAND_RED   = "#FFC9C9"   #  0 到 1
BAND_ALPHA = 0.72

FS_XTICK  = 18
FS_YTICK  = 17
FS_LEGEND = 12

# 年份数字和连接线
YEAR_FS = 10.8
YEAR_FS_2025 = 13.0
YEAR_CONNECT_LW = 1.45
YEAR_CONNECT_LW_2025 = 2.00

N = len(labels)

# 背景图层：粉色、蓝色背景和白色 0 线使用同一层
Z_BACKGROUND = -3


# ===============================
# Helpers
# ===============================
def clip_vals(vals):
    vals = pd.to_numeric(pd.Series(vals), errors="coerce").values.astype(float)
    vals = np.nan_to_num(vals, nan=V_MIN)
    return np.clip(vals, V_MIN, V_MAX)


def normalize(vals):
    vals_used = clip_vals(vals)
    r = (vals_used - V_MIN) / (V_MAX - V_MIN)
    return r, vals_used


def value_to_r(value, v_min=V_MIN, v_max=V_MAX):
    return (value - v_min) / (v_max - v_min)


def get_angles(n):
    return np.linspace(0, 2 * np.pi, n, endpoint=False)


def close_theta(theta):
    return np.r_[theta, theta[0] + 2 * np.pi]


def close_r(r):
    return np.r_[r, r[0]]


def plot_line(ax, theta, r, color, lw, ls="-", z=3, alpha=1.0):
    ax.plot(
        close_theta(theta),
        close_r(r),
        color=color,
        lw=lw,
        linestyle=ls,
        zorder=z,
        alpha=alpha,
        solid_capstyle="round",
        solid_joinstyle="round"
    )


def radar_edge_point_cartesian(theta, r, segment_index, frac):
    """
    在雷达图五边形的真实边上取点。

    重要：
    雷达图五边形边是两个顶点在图面上的直线连接。
    因此不能简单做极坐标插值：
        theta 线性插值 + r 线性插值
    那样得到的点不一定落在视觉上的五边形边上。

    正确做法：
    1. 把边两端顶点从极坐标转成笛卡尔坐标；
    2. 在笛卡尔坐标中沿直线段插值；
    3. 再把该点转回极坐标。
    """

    n = len(theta)

    i = segment_index % n
    j = (segment_index + 1) % n

    th_i = theta[i]
    th_j = theta[j]

    r_i = r[i]
    r_j = r[j]

    # 极坐标 -> 笛卡尔坐标
    x_i = r_i * np.cos(th_i)
    y_i = r_i * np.sin(th_i)

    x_j = r_j * np.cos(th_j)
    y_j = r_j * np.sin(th_j)

    # 在真实直线边上插值
    x = x_i + frac * (x_j - x_i)
    y = y_i + frac * (y_j - y_i)

    # 笛卡尔坐标 -> 极坐标
    th = np.arctan2(y, x)
    rr = np.sqrt(x ** 2 + y ** 2)

    # 保持角度为 0 到 2π，便于 polar axes 使用
    if th < 0:
        th += 2 * np.pi

    return th, rr


def polar_text_align(theta_val):
    """
    根据极坐标角度自动设置文字对齐方式。
    """

    angle = np.degrees(theta_val) % 360

    if 90 < angle < 270:
        ha = "right"
    elif np.isclose(angle, 90) or np.isclose(angle, 270):
        ha = "center"
    else:
        ha = "left"

    if 0 < angle < 180:
        va = "bottom"
    elif 180 < angle < 360:
        va = "top"
    else:
        va = "center"

    return ha, va


def annotate_event_year(
    ax,
    theta,
    r,
    year,
    color,
    offset_index=0,
    is_2025=False,
    z=30
):
    """
    为单个雷达五边形事件添加年份标注。

    核心修正：
    - 连接线起点使用 radar_edge_point_cartesian()；
    - 该点位于雷达图真实五边形边上；
    - 解决“不规则五边形引导线没有接触边”的问题。
    """

    n = len(theta)

    # 不同年份从不同边拉出
    segment_index = offset_index % n

    # 不同年份从边上的不同位置拉出
    frac_choices = [0.10, 0.24, 0.38, 0.52, 0.66, 0.80, 0.92]
    frac = frac_choices[(offset_index // n) % len(frac_choices)]

    # 关键：从雷达图真实边上取点，而不是极坐标线性插值
    th_anchor, r_anchor = radar_edge_point_cartesian(
        theta=theta,
        r=r,
        segment_index=segment_index,
        frac=frac
    )

    # 标签角度分散
    angle_shift_choices = [-14, -10, -6, -2, 2, 6, 10, 14]
    angle_shift = np.deg2rad(
        angle_shift_choices[offset_index % len(angle_shift_choices)]
    )

    th_label = th_anchor + angle_shift

    # 标签半径分层
    outer_levels = [1.10, 1.18, 1.26, 1.34, 1.42]
    r_label = outer_levels[offset_index % len(outer_levels)]

    if r_anchor > 0.92 and not is_2025:
        r_label = min(1.45, r_label + 0.04)

    if is_2025:
        r_label = 1.47

    ha, va = polar_text_align(th_label)

    # 连接线：起点严格在雷达五边形边上
    ax.plot(
        [th_anchor, th_label],
        [r_anchor, r_label - 0.035],
        color=color,
        lw=YEAR_CONNECT_LW if not is_2025 else YEAR_CONNECT_LW_2025,
        linestyle="-",
        alpha=0.98 if not is_2025 else 1.0,
        zorder=z,
        solid_capstyle="round"
    )

    # 起点小圆点：让连接位置一眼可见
    ax.scatter(
        [th_anchor],
        [r_anchor],
        s=30 if not is_2025 else 50,
        color=color,
        edgecolor="white",
        linewidth=0.9 if not is_2025 else 1.2,
        alpha=1.0,
        zorder=z + 1
    )

    # 年份数字
    ax.text(
        th_label,
        r_label,
        str(int(year)),
        fontsize=YEAR_FS if not is_2025 else YEAR_FS_2025,
        fontweight="normal" if not is_2025 else "bold",
        color=color,
        ha=ha,
        va=va,
        zorder=z + 2,
        bbox=dict(
            boxstyle="round,pad=0.24",
            facecolor="white",
            edgecolor=color if is_2025 else "none",
            linewidth=1.0 if is_2025 else 0.0,
            alpha=0.92 if not is_2025 else 0.97
        )
    )


# ===============================
# Background Bands
# ===============================
def draw_value_band(ax, theta, v_low, v_high, color, alpha=0.6, z=Z_BACKGROUND):
    r_low = value_to_r(v_low)
    r_high = value_to_r(v_high)

    r_low_arr = np.full(len(theta), r_low)
    r_high_arr = np.full(len(theta), r_high)

    ax.fill_between(
        close_theta(theta),
        close_r(r_low_arr),
        close_r(r_high_arr),
        color=color,
        alpha=alpha,
        linewidth=0,
        zorder=z
    )


# ===============================
# Density Envelope
# ===============================
def draw_density_envelope(
    ax,
    theta,
    r_matrix,
    base_color,
    n_bins=DENSITY_BINS,
    alpha_base=ALPHA_BASE,
    z=1
):
    n_years, n_axes = r_matrix.shape
    r_lo_global = r_matrix.min()
    r_hi_global = r_matrix.max()

    if np.isclose(r_lo_global, r_hi_global):
        return

    bins = np.linspace(r_lo_global, r_hi_global, n_bins + 1)
    rgb = mcolors.to_rgb(base_color)

    for b in range(n_bins):
        lo = bins[b]
        hi = bins[b + 1]

        pass_count = ((r_matrix >= lo) & (r_matrix < hi)).sum(axis=0)
        avg_pass = pass_count.mean()

        if avg_pass < 0.5:
            continue

        alpha = min(alpha_base * avg_pass, 0.60)

        r_lo_arr = np.full(n_axes, lo)
        r_hi_arr = np.full(n_axes, hi)

        ax.fill_between(
            close_theta(theta),
            close_r(r_lo_arr),
            close_r(r_hi_arr),
            color=rgb,
            alpha=alpha,
            linewidth=0,
            zorder=z
        )


# ===============================
# Grid
# ===============================
def draw_grid(ax, theta, v_min, v_max, r_spoke_max_per_axis):
    ax.set_yticks([])
    ax.yaxis.grid(False)
    ax.xaxis.grid(False)
    ax.spines["polar"].set_visible(False)

    zero_r = value_to_r(0, v_min, v_max)

    th_closed = np.append(theta, theta[0])
    r_closed  = np.full(len(th_closed), zero_r)

    # 白色 0 线：与粉色、蓝色背景使用同一图层
    ax.plot(
        th_closed,
        r_closed,
        color="white",
        lw=3.0,
        linestyle="-",
        zorder=Z_BACKGROUND,
        alpha=1.0
    )

    # 0 标签：同样放在背景图层
    ax.text(
        theta[0],
        zero_r + 0.03,
        "0",
        fontsize=FS_YTICK,
        fontweight="bold",
        color="white",
        ha="center",
        va="bottom",
        zorder=Z_BACKGROUND
    )

    # 辐条
    for i, th in enumerate(theta):
        ax.plot(
            [th, th],
            [0, r_spoke_max_per_axis[i]],
            color="#CCCCCC",
            lw=0.85,
            linestyle="-",
            zorder=0,
            alpha=0.58
        )


def set_axis_labels(ax, theta, labels, r_outer, pad=0.30):
    ax.set_xticks([])

    for th, lbl in zip(theta, labels):
        ax.text(
            th,
            r_outer + pad,
            lbl,
            fontsize=FS_XTICK,
            fontweight="normal",
            ha="center",
            va="center",
            color="#333333",
            zorder=12
        )


# ===============================
# Main
# ===============================
if __name__ == "__main__":
    df = pd.read_csv(data_path)

    years    = pd.to_numeric(df[YEAR_COL],    errors="coerce").astype("Int64")
    spi      = pd.to_numeric(df[SPI3_COL],    errors="coerce")
    clusters = pd.to_numeric(df[CLUSTER_COL], errors="coerce").astype("Int64")

    # ---- 2025 ----
    mask_2025 = years == 2025

    if not mask_2025.any():
        raise ValueError("No data for 2025.")

    vals_2025 = df.loc[mask_2025, csv_cols].mean().values

    cluster_2025 = int(
        clusters.loc[mask_2025].dropna().mode().iloc[0]
    )

    color_2025 = cluster_colors[cluster_2025]
    roman_2025 = cluster_roman[cluster_2025]

    # ---- Historical Events: SPI-3 < -1, excluding 2025 ----
    mask_hist = (
        (spi < -1)
        & (~mask_2025)
        & years.notna()
        & clusters.notna()
    )

    hist = df.loc[
        mask_hist,
        [YEAR_COL, CLUSTER_COL] + csv_cols
    ].copy()

    year_mean = hist.groupby(YEAR_COL)[csv_cols].mean()

    year_cluster = hist.groupby(YEAR_COL)[CLUSTER_COL].agg(
        lambda x: int(x.mode().iloc[0])
    )

    sorted_clusters = sorted(cluster_colors.keys())

    # ---- 每个聚类的统计：仅用于密度色块 ----
    stats = {}

    for k in sorted_clusters:
        yrs = year_cluster[year_cluster == k].index

        if len(yrs) == 0:
            continue

        mat = np.vstack([
            normalize(row)[0]
            for row in year_mean.loc[yrs].values
        ])

        stats[k] = dict(
            count=len(yrs),
            years=list(yrs),
            r_matrix=mat,
            color=cluster_colors[k]
        )

    # ---- 辐条最大值：历史事件 + 2025 ----
    all_r = np.vstack(
        [normalize(row)[0] for row in year_mean.values]
        + [normalize(vals_2025)[0]]
    )

    r_spoke_max = all_r.max(axis=0)

    # ---- Plot ----
    fig = plt.figure(figsize=(12, 10), facecolor="white")
    ax  = fig.add_subplot(111, projection="polar")

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    # 外圈范围加大，给年份标注留足空间
    ax.set_ylim(0, 1.52)

    theta = get_angles(N)

    # 0. 背景环带
    draw_value_band(
        ax=ax,
        theta=theta,
        v_low=-1,
        v_high=0,
        color=BAND_BLUE,
        alpha=BAND_ALPHA,
        z=Z_BACKGROUND
    )

    draw_value_band(
        ax=ax,
        theta=theta,
        v_low=0,
        v_high=1,
        color=BAND_RED,
        alpha=BAND_ALPHA,
        z=Z_BACKGROUND
    )

    # 1. Grid & Labels
    draw_grid(
        ax=ax,
        theta=theta,
        v_min=V_MIN,
        v_max=V_MAX,
        r_spoke_max_per_axis=r_spoke_max
    )

    set_axis_labels(
        ax=ax,
        theta=theta,
        labels=labels,
        r_outer=r_spoke_max.max(),
        pad=0.30
    )

    # 2. 密度色块
    for k in sorted_clusters:
        if k not in stats or stats[k]["count"] < 2:
            continue

        draw_density_envelope(
            ax=ax,
            theta=theta,
            r_matrix=stats[k]["r_matrix"],
            base_color=stats[k]["color"],
            z=1
        )

    # 3. 每个历史干旱事件逐年绘制为实线，并分散标注年份
    sorted_years = sorted(year_mean.index)

    for idx_yr, yr in enumerate(sorted_years):
        k = int(year_cluster.loc[yr])

        if k not in cluster_colors:
            continue

        r_event, _ = normalize(year_mean.loc[yr].values)

        plot_line(
            ax=ax,
            theta=theta,
            r=r_event,
            color=cluster_colors[k],
            lw=EVENT_LW,
            ls="-",
            z=6,
            alpha=EVENT_ALPHA
        )

        annotate_event_year(
            ax=ax,
            theta=theta,
            r=r_event,
            year=yr,
            color=cluster_colors[k],
            offset_index=idx_yr,
            is_2025=False,
            z=30
        )

    # 4. 2025 也作为普通事件绘制，并单独标注
    r2025, _ = normalize(vals_2025)

    plot_line(
        ax=ax,
        theta=theta,
        r=r2025,
        color=color_2025,
        lw=EVENT_LW,
        ls="-",
        z=7,
        alpha=1.0
    )

    annotate_event_year(
        ax=ax,
        theta=theta,
        r=r2025,
        year=2025,
        color=color_2025,
        offset_index=len(sorted_years) + 3,
        is_2025=True,
        z=34
    )

    # ==========================================
    # Legend
    # ==========================================
    legend_handles = []

    for k in sorted_clusters:
        if k not in stats:
            continue

        c     = cluster_colors[k]
        roman = cluster_roman[k]
        n_yr  = stats[k]["count"]

        legend_handles.append(
            Line2D(
                [0], [0],
                color=c,
                lw=EVENT_LW,
                linestyle="-",
                label=f"Cluster {roman} events (n={n_yr})"
            )
        )

        legend_handles.append(
            Patch(
                facecolor=c,
                edgecolor="none",
                alpha=0.55,
                label=f"Cluster {roman} distribution (darker = denser)"
            )
        )

    legend_handles.append(
        Patch(
            facecolor=BAND_BLUE,
            edgecolor="none",
            alpha=BAND_ALPHA,
            label="-1 to 0 background"
        )
    )

    legend_handles.append(
        Patch(
            facecolor=BAND_RED,
            edgecolor="none",
            alpha=BAND_ALPHA,
            label="0 to 1 background"
        )
    )

    legend_handles.append(
        Line2D(
            [0], [0],
            color=color_2025,
            lw=EVENT_LW,
            linestyle="-",
            label=f"2025 event (Cluster {roman_2025})"
        )
    )

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.50, 0.00),
        ncol=3,
        frameon=False,
        fontsize=FS_LEGEND,
        handlelength=2.4,
        handleheight=1.1,
        columnspacing=1.4,
        labelspacing=0.6
    )

    fig.subplots_adjust(top=0.94, bottom=0.16)

    fig.savefig(out_png, dpi=360, bbox_inches="tight", facecolor="white")
    fig.savefig(out_pdf, bbox_inches="tight", facecolor="white")

    plt.close(fig)

    print("✅ Done.")
    print(f"PNG: {out_png}")
    print(f"PDF: {out_pdf}")
